<a href="https://colab.research.google.com/github/olexvysotina-cell/ML/blob/main/%22%D0%9B%D0%A0_9_%D0%A0%D0%B5%D0%BA%D0%BE%D0%BC%D0%B5%D0%BD%D0%B4%D0%B0%D1%86%D1%96%D0%B9%D0%BD%D1%96_%D1%81%D0%B8%D1%81%D1%82%D0%B5%D0%BC%D0%B8_%D0%92%D0%B8%D1%81%D0%BE%D1%82%D1%96%D0%BD%D0%B0_12_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** Висотіна Олександра Володимирівна  
**Група:** 3-12  
**Дата:** 10.11.2026

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [2]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [3]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.18
NumPy: 2.4.6
SciPy: 1.15.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [4]:
# Завантаження даних (готова частина шаблону)
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k
Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [5]:
# TODO 2.1 — Дослідіть набір даних:
# - кількість унікальних користувачів;
# - кількість унікальних фільмів;
# - мінімальне, максимальне, середнє значення rating;
# - пропущені значення (за наявності).

# Ваш код:
print("Унікальні користувачі:", ratings_df["user_id"].nunique())
print("Унікальні фільми:", ratings_df["item_id"].nunique())
print("Мінімальне значення:", ratings_df["rating"].min())
print("Максимальне значення:", ratings_df["rating"].max())
print("Середнє значення:", ratings_df["rating"].mean())
print("Пропущенні значення:")
print(ratings_df.isna().sum())

Унікальні користувачі: 943
Унікальні фільми: 1682
Мінімальне значення: 1.0
Максимальне значення: 5.0
Середнє значення: 3.52986
Пропущенні значення:
user_id      0
item_id      0
rating       0
timestamp    0
dtype: int64


**Короткі спостереження щодо даних:**

_Запишіть 2–3 речення про кількість користувачів, фільмів та характер оцінок._
Набір даних містить 943 унікальних користувачів і 1682 унікальних фільмів. Мінімальний рейтинг фільму - 1, максимальний рейтинг фільм - 5. Середня оцінка 3.52986, що свідчить про дещо вищу оцінку фільмів користувачами.

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [6]:
# TODO 3.1 — Створіть train_df та test_df із ratings_df.
# Використайте sklearn_train_test_split(..., test_size=0.2, random_state=42).

# Ваш код:
train_df, test_df = sklearn_train_test_split(
    ratings_df,
    test_size = 0.2,
    random_state = 42
)

print("Розмір навчальної вибірки:", train_df.shape)
print("Розмір тестової вибірки:", test_df.shape)

print("\nЧастка навчальної вибірки:", len(train_df) / len(ratings_df))
print("Частка тестової вибірки:", len(test_df) / len(ratings_df))

Розмір навчальної вибірки: (80000, 4)
Розмір тестової вибірки: (20000, 4)

Частка навчальної вибірки: 0.8
Частка тестової вибірки: 0.2


In [7]:
# TODO 3.2 — Підготуйте об’єкти для Surprise:
# 1. reader = Reader(rating_scale=...)
#    Підказка: data.reader.rating_scale
# 2. train_data = Dataset.load_from_df(...)
#    Візьміть тільки user_id, item_id, rating із train_df.
# 3. testset = список кортежів (user_id, item_id, rating) із test_df.
# 4. Перевірте розміри train/test.

# Ваш код:
reader = Reader(rating_scale=data.reader.rating_scale)

train_data = Dataset.load_from_df(train_df[["user_id", "item_id", "rating"]], reader)

testset = list(
    test_df[["user_id", "item_id", "rating"]].itertuples(
        index=False,
        name=None
    )
)

print("Кількість записів у train:", len(train_df))
print("Кількість записів у test:", len(testset))

print("Розмір train:", train_df.shape)
print("Розмір test:", len(testset))

Кількість записів у train: 80000
Кількість записів у test: 20000
Розмір train: (80000, 4)
Розмір test: 20000


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [19]:
# Стартові сітки параметрів (можна змінювати)
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

**Обґрунтування вибору параметрів:**

_Поясніть, які параметри впливають на якість моделей і чому обрали саме ці значення._

Для моделі SVD було обрано кількість прихованих факторів (n_factors) — 50 та 100, швидкість навчання (lr_all) — 0.005 та 0.010, а також коефіцієнт регуляризації (reg_all) — 0.02 та 0.05. Кількість прихованих факторів впливає на здатність моделі виявляти приховані закономірності в оцінках користувачів. Швидкість навчання визначає темп оновлення параметрів моделі, а регуляризація допомагає запобігти перенавчанню.

Для моделі KNNBasic було обрано кількість сусідів (k) — 20 та 40, мінімальну кількість сусідів (min_k) — 1 та 5, метрики схожості (cosine та pearson), а також два підходи до пошуку схожості: між користувачами та між товарами. Ці параметри впливають на визначення схожості об'єктів і формування прогнозованих оцінок.

Вибрані значення дозволяють порівняти різні налаштування моделей та визначити найкращу комбінацію гіперпараметрів за результатами подальшого пошуку.

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [20]:
# TODO 5.1 — Підберіть гіперпараметри SVD.
# Підказка: GridSearchCV(SVD, param_grid_svd,
#                        measures=['rmse', 'mae'], cv=3, n_jobs=-1)
# Після створення викличте .fit(train_data).

# Ваш код:
grid_search_svd = GridSearchCV(
    SVD,
    param_grid_svd,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)

grid_search_svd.fit(train_data)

In [21]:
# TODO 5.2 — Підберіть гіперпараметри KNNBasic.
# Підказка: використайте param_grid_knn та ті самі метрики/cv.

# Ваш код:
grid_search_knn = GridSearchCV(
    KNNBasic,
    param_grid_knn,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)

grid_search_knn.fit(train_data)

In [32]:
# TODO 5.3 — Виведіть найкращий RMSE, MAE та параметри обох моделей.
# Підказки:
# grid_search_svd.best_score['rmse']
# grid_search_svd.best_params['rmse']
# grid_search_knn.best_score['rmse']

# Ваш код:
print("Результати SVD:")
print("Найкращий RMSE:", grid_search_svd.best_score['rmse'])
print("Параметри RMSE:", grid_search_svd.best_params['rmse'])
print("Найкращий MAE:", grid_search_svd.best_score['mae'])
print("Параметри MAE:", grid_search_svd.best_params['mae'])

print("\nРезультати KNNBasic:")
print("Найкращий RMSE:", grid_search_knn.best_score['rmse'])
print("Параметри RMSE:", grid_search_knn.best_params['rmse'])
print("Найкращий MAE:", grid_search_knn.best_score['mae'])
print("Параметри MAE:", grid_search_knn.best_params['mae'])

Результати SVD:
Найкращий RMSE: 0.9453385730398253
Параметри RMSE: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}
Найкращий MAE: 0.7456434954723484
Параметри MAE: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}

Результати KNNBasic:
Найкращий RMSE: 1.0276099589538965
Параметри RMSE: {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}
Найкращий MAE: 0.8152351842008289
Параметри MAE: {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}


**Аналіз результатів кросвалідації:**

| Модель | Найкращий CV RMSE | CV MAE | Найкращі параметри за RMSE |
|:--|--:|--:|:--|
| SVD | ... | ... | ... |
| KNNBasic | ... | ... | ... |

_Порівняйте якість моделей на перехресній перевірці. Зазначте, які параметри стали найкращими._

За результатами перехресної перевірки за допомогою GridSearchCV було встановлено, що модель SVD демонструє кращу якість прогнозування порівняно з KNNBasic.

Для моделі SVD найкраще значення RMSE становить 0.9453, а MAE — 0.7456. Оптимальними параметрами стали n_factors = 50, lr_all = 0.01 та reg_all = 0.05.

Для моделі KNNBasic найкраще значення RMSE становить 1.0276, а MAE — 0.8152. Найкращими параметрами виявилися k = 40, min_k = 1, метрика схожості cosine та user_based = True.

Порівняння результатів показало, що SVD має менші значення обох метрик похибки. Отже, серед досліджених моделей SVD забезпечила найкращу якість прогнозування оцінок користувачів.

## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [23]:
# TODO 6.1 — Отримайте best_svd_params і best_knn_params за CV RMSE.
# TODO 6.2 — Створіть trainset із train_data.
# TODO 6.3 — Навчіть best_svd і best_knn.
# Для відтворюваності задайте random_state=42 для SVD.

# Ваш код:
best_svd_params = grid_search_svd.best_params['rmse']
best_knn_params = grid_search_knn.best_params['rmse']

print("Найкращі параметри SVD:", best_svd_params)
print("Найкращі параметри KNNBasic:", best_knn_params)

Найкращі параметри SVD: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}
Найкращі параметри KNNBasic: {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}


In [24]:
trainset = train_data.build_full_trainset()

In [25]:
best_svd = SVD(**best_svd_params, random_state=42)
best_knn = KNNBasic(**best_knn_params)

best_svd.fit(trainset)
best_knn.fit(trainset)

Computing the cosine similarity matrix...
Done computing similarity matrix.


## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [26]:
# TODO 7.1 — Отримайте svd_predictions та knn_predictions.
# TODO 7.2 — Обчисліть svd_rmse, svd_mae, knn_rmse, knn_mae.
# Підказка: accuracy.rmse(predictions, verbose=False)
#           accuracy.mae(predictions, verbose=False)

# Ваш код:
svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)

In [27]:
svd_rmse = accuracy.rmse(svd_predictions, verbose=False)
svd_mae = accuracy.mae(svd_predictions, verbose=False)

knn_rmse = accuracy.rmse(knn_predictions, verbose=False)
knn_mae = accuracy.mae(knn_predictions, verbose=False)

print("Результати SVD:")
print("RMSE:", svd_rmse)
print("MAE:", svd_mae)

print("\nРезультати KNNBasic:")
print("RMSE:", knn_rmse)
print("MAE:", knn_mae)

Результати SVD:
RMSE: 0.9191746414249822
MAE: 0.7233554064545319

Результати KNNBasic:
RMSE: 1.0119527444069378
MAE: 0.7984207578945732


In [28]:
# TODO 7.3 — Сформуйте DataFrame results із колонками:
# Model, RMSE, MAE.
# Відсортуйте за RMSE та покажіть порівняння.

# Ваш код:
results = pd.DataFrame({
    'Model': ['SVD', 'KNNBasic'],
    'RMSE': [svd_rmse, knn_rmse],
    'MAE': [svd_mae, knn_mae]
})

results = results.sort_values(by='RMSE').reset_index(drop=True)

display(results)

,Model,RMSE,MAE
0,SVD,0.919175,0.723355
1,KNNBasic,1.011953,0.798421


**Таблиця тестових результатів:**

| Модель | Test RMSE | Test MAE |
|:--|--:|--:|
| SVD | ... | ... |
| KNNBasic | ... | ... |

**Порівняння з CV:** _Прокоментуйте, чи суттєво відрізняються оцінки на кросвалідації та незалежній тестовій вибірці._

Результати кросвалідації та незалежної тестової вибірки суттєво не відрізняються. Для обох моделей значення RMSE та MAE на тестовій вибірці дещо нижчі, ніж під час кросвалідації. Це свідчить про стабільність отриманих результатів і відсутність суттєвого погіршення якості прогнозування на нових даних. Модель SVD зберегла кращі результати за обома метриками порівняно з KNNBasic.

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [38]:
# Початковий ID можна змінити
user_id = '196'

# TODO 8.1 — Визначте найкращу за CV RMSE модель.
# TODO 8.2 — Перевірте, що користувач присутній у trainset.
# TODO 8.3 — Знайдіть неоцінені ним фільми.
# TODO 8.4 — Спрогнозуйте оцінки та покажіть топ-10.
# Бажані колонки результату: item_id, predicted_rating.
# Передбачте ситуацію, коли обраного user_id немає у trainset.

# Ваш код:
  # TODO 8.1
if grid_search_svd.best_score['rmse'] < grid_search_knn.best_score['rmse']:
    best_model = best_svd
    print("Обрана модель: SVD")
else:
    best_model = best_knn
    print("Обрана модель: KNNBasic")


  # TODO 8.2
try:
    user_inner_id = trainset.to_inner_uid(user_id)

  # TODO 8.3

    all_items = trainset.all_items()

    rated_items = set(
        item_id for (item_id, rating) in trainset.ur[user_inner_id]
    )

    unrated_items = [
        item_id for item_id in all_items
        if item_id not in rated_items
    ]

    # TODO 8.4

    predictions = []

    for item_inner_id in unrated_items:

        item_raw_id = trainset.to_raw_iid(item_inner_id)

        predicted_rating = best_model.predict(
            uid=user_id,
            iid=item_raw_id
        ).est

        predictions.append({
            'item_id': item_raw_id,
            'predicted_rating': predicted_rating
        })

    recommendations = pd.DataFrame(predictions)

    recommendations = recommendations.sort_values(
        by='predicted_rating',
        ascending=False
    ).head(10).reset_index(drop=True)

    display(recommendations)

except ValueError:
    print(f"Користувача з ID {user_id} немає у навчальній вибірці.")

Обрана модель: SVD


,item_id,predicted_rating
0,320,4.490358
1,64,4.470680
2,480,4.413710
3,313,4.401920
4,513,4.399624
5,483,4.398092
6,318,4.392102
7,656,4.385379
8,22,4.377388
9,272,4.376010


## 9. Перевірка виконання

Позначте після реалізації:

- [ *] Дані `ml-100k` завантажено та перевірено.
- [ *] Є розбиття 80/20 з фіксованим `random_state`.
- [ *] `Reader`, `train_data` і `testset` створені коректно.
- [ *] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [ *] Для SVD і KNNBasic показано параметри та результати CV.
- [ *] Моделі навчено на повній навчальній частині.
- [ *] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [ *] Наведено порівняльну таблицю та інтерпретацію.
- [ *] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [ *] Усі комірки виконуються послідовно після перезапуску середовища.

## 10. Висновки

Заповніть **власними результатами**:

1. Яку модель обрано за кросвалідацією і чому?
2. Які найкращі гіперпараметри отримано для обох моделей?
3. Які підсумкові значення RMSE та MAE на тестовій вибірці?
4. Чи є різниця між результатами CV та тестування?
5. Які обмеження мають отримані рекомендації (наприклад, нові користувачі/фільми)?

**Висновок:** _Напишіть 5–8 речень з опорою на отримані числові результати._

---
**Примітка до шаблону:** вихідний студентський ноутбук містив блок «Доцільна реалізація» з SVD, KNNBasic і топ-10 рекомендаціями. Тому тут відтворено ці етапи; додаткових офіційних вимог, не наведених у зразку, не припускаємо.

Висновки:
1. За результатами кросвалідації найкращою обрано модель SVD, оскільки вона продемонструвала менші значення RMSE = 0,9453 та MAE = 0,7456 порівняно з KNNBasic (RMSE = 1,0276; MAE = 0,8152).
2. Для моделі SVD найкращими гіперпараметрами виявилися n_factors = 50, lr_all = 0.01, reg_all = 0.05. Для моделі KNNBasic оптимальними стали k = 40, min_k = 1, sim_options = {'name': 'cosine', 'user_based': True}.
3. На тестовій вибірці модель SVD отримала значення RMSE - 0.9191746414249822, MAE - 0.7233554064545319. Модель KNN отримала значення RMSE - 1.011953, MAE - 0.798421.
4. Суттєвої різниці між результатами не виявлено, проте значення RMSE та MAE на тестовій вибірці виявилися дещо нижчими, ніж під час кросвалідації, що свідчить про відсутність погіршення якості моделі на тестових даних.
5. За допомогою обраної моделі SVD також було сформовано персоналізований список із 10 рекомендованих фільмів, найвищий прогнозований рейтинг серед яких становив 4,4904. Основними обмеженнями рекомендаційної системи є проблема холодного старту для нових користувачів і фільмів, які ще не мають оцінок, а також залежність якості рекомендацій від кількості та повноти наявних оцінок користувачів.

**Додаткове завдання. Написати власний рекомендаційний алгоритм з нуля**